<a href="https://colab.research.google.com/github/Ash100/AIMe/blob/main/Cla_new_FEL_Analysis_4Complexes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Free Energy Landscape (FEL) Analysis
## Comparative Analysis of 4 Receptor–De Novo Binder Complexes

**Method:** PCA on receptor + interface contact residues → KDE → FEL  
**Input:** OpenMM trajectories (.dcd) + topology (.pdb)  
**Output:** Publication-quality FEL plots + lowest-energy basin structures  

> **PCA basis:** receptor backbone Cα + per-complex interface residues (within 5 Å of binder). Comparable across all 4 systems despite differing binder lengths.

## Step 1 — Install Dependencies

In [3]:
%%capture
!pip install mdtraj numpy scipy matplotlib scikit-learn nglview ipywidgets pandas
print('Dependencies installed.')

## Step 2 — Mount Google Drive and Configure Paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# ── Define exact paths for each complex individually ──

COMPLEXES_RAW = {
    'Complex A': {
        'pdb': '/content/drive/MyDrive/PagC/PagC1_Repeat/PagC1_equil.pdb',
        'dcd': '/content/drive/MyDrive/PagC/PagC1_Repeat/PagC1_prod1-10_whole.dcd',
    },
    'Complex B': {
        'pdb': '/content/drive/MyDrive/PagC/PagC2/PagC2_equil.pdb',
        'dcd': '/content/drive/MyDrive/PagC/PagC2/PagC2_prod1-10_whole.dcd',
    },
    'Complex C': {
        'pdb': '/content/drive/MyDrive/PagC/PagC3/PagC3_equil.pdb',
        'dcd': '/content/drive/MyDrive/PagC/PagC3/PagC3_prod1-10_whole.dcd',
    },
    'Complex D': {
        'pdb': '/content/drive/MyDrive/PagC/PagC4_equil.pdb',
        'dcd': '/content/drive/MyDrive/PagC/PagC4_prod1-10_whole.dcd',
    },
}

# Clean each PDB and build the COMPLEXES dict the rest of the notebook uses
import os

COMPLEXES = {}
for label, paths in COMPLEXES_RAW.items():
    src = paths['pdb']
    dst = src.replace('.pdb', '_clean.pdb')   # saved alongside original
    print(f'\nCleaning {label} ...')
    if not os.path.exists(src):
        print(f'  ❌ Not found: {src}')
        continue
    clean_pdb(src, dst)
    COMPLEXES[label] = (dst, paths['dcd'])

print('\nCOMPLEXES ready:')
for label, (pdb, dcd) in COMPLEXES.items():
    print(f'  {label}: {pdb}')
    print(f'          {dcd}')

## Step 3 — Load Trajectories and Detect Interface Residues

In [25]:
import mdtraj as md
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ADD THIS IMPORT:
try:
    import openmm.unit as openmm_unit
except ImportError:
    # For older versions of OpenMM
    import simtk.unit as openmm_unit

In [27]:
import mdtraj as md
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Receptor is residues 1-162 (by actual PDB residue number) in every complex.
# Everything with a higher residue number is the binder.
RECEPTOR_MAX_RESSEQ = 162
INTERFACE_CUTOFF_NM = 0.5   # 5 Å — plain float, no openmm_unit needed

def load_trajectory(pdb_path, dcd_path):
    traj = md.load(dcd_path, top=pdb_path)
    traj = traj.atom_slice(traj.top.select('protein'))
    print(f'  Frames: {traj.n_frames:,}  Atoms: {traj.n_atoms:,}  Residues: {traj.n_residues:,}')
    return traj

def get_interface_indices(traj, receptor_max_resseq, cutoff_nm=0.5):
    """
    Select receptor vs binder using ACTUAL PDB residue numbers (resSeq),
    not mdtraj's internal 0-based 'resid'. This is robust to numbering gaps,
    missing loops, or chains that share the same chain ID/letter.

    receptor = all atoms with resSeq <= receptor_max_resseq
    binder   = all atoms with resSeq >  receptor_max_resseq
    """
    top = traj.topology
    resseq_per_atom = np.array([atom.residue.resSeq for atom in top.atoms])
    is_ca = np.array([atom.name == 'CA' for atom in top.atoms])

    rec_ca = np.where(is_ca & (resseq_per_atom <= receptor_max_resseq))[0]
    binder = np.where(resseq_per_atom > receptor_max_resseq)[0]

    if len(rec_ca) == 0:
        raise ValueError(
            f'No receptor CA atoms found with resSeq <= {receptor_max_resseq}. '
            'Check RECEPTOR_MAX_RESSEQ against the actual PDB numbering.'
        )
    if len(binder) == 0:
        raise ValueError(
            f'No binder atoms found with resSeq > {receptor_max_resseq}. '
            'Check RECEPTOR_MAX_RESSEQ against the actual PDB numbering.'
        )

    print(f'  Receptor CA: {len(rec_ca)}   Binder heavy atoms: {len(binder)}')

    min_dist = np.full(len(rec_ca), np.inf)
    BATCH = 300
    for b0 in range(0, len(binder), BATCH):
        b_batch = binder[b0:b0+BATCH]
        pairs = np.array([[r, b] for r in rec_ca for b in b_batch])
        dists = md.compute_distances(traj, pairs).reshape(
            traj.n_frames, len(rec_ca), len(b_batch))
        min_dist = np.minimum(min_dist, dists.min(axis=(0, 2)))

    iface_ca = rec_ca[min_dist < cutoff_nm]
    print(f'  Interface residues (< {cutoff_nm*10:.0f} A): {len(iface_ca)}')
    return rec_ca, iface_ca

trajs, rec_ca_idx, iface_ca_idx = {}, {}, {}
for label, (pdb, dcd) in COMPLEXES.items():
    print(f'\nLoading {label} ...')
    t = load_trajectory(pdb, dcd)
    rec, iface = get_interface_indices(t, RECEPTOR_MAX_RESSEQ, INTERFACE_CUTOFF_NM)
    trajs[label]        = t
    rec_ca_idx[label]   = rec
    iface_ca_idx[label] = iface
print('\nAll trajectories loaded.')


Loading Complex A ...


NameError: name 'openmm_unit' is not defined

## Step 4 — PCA on Receptor + Interface Coordinates

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def run_pca(traj, ca_idx):
    traj.superpose(traj, frame=0, atom_indices=ca_idx)
    coords = traj.xyz[:, ca_idx, :].reshape(traj.n_frames, -1)
    X = StandardScaler().fit_transform(coords)
    pca = PCA(n_components=2, random_state=42)
    pcs = pca.fit_transform(X)
    return pcs, pca.explained_variance_ratio_ * 100

pca_results = {}
for label, traj in trajs.items():
    print(f'\nPCA for {label} ...')
    ca = iface_ca_idx[label]
    if len(ca) < 3:
        print('  Too few interface residues; using all receptor CA.')
        ca = rec_ca_idx[label]
    pcs, ev = run_pca(traj, ca)
    print(f'  PC1: {ev[0]:.1f}%   PC2: {ev[1]:.1f}%   Total: {ev.sum():.1f}%')
    pca_results[label] = {'pc1': pcs[:,0], 'pc2': pcs[:,1], 'ev': ev, 'n_iface': len(ca)}
print('\nPCA complete.')

## Step 5 — Free Energy Landscape via KDE

In [ ]:
from scipy.stats import gaussian_kde

kT = 1.380649e-23 * 6.02214076e23 * TEMPERATURE_K / 1000  # kJ/mol
GRID_BINS = 150

def compute_FEL(pc1, pc2, bins=GRID_BINS):
    pad1 = (pc1.max() - pc1.min()) * 0.05
    pad2 = (pc2.max() - pc2.min()) * 0.05
    xg = np.linspace(pc1.min()-pad1, pc1.max()+pad1, bins)
    yg = np.linspace(pc2.min()-pad2, pc2.max()+pad2, bins)
    X, Y = np.meshgrid(xg, yg)
    kde  = gaussian_kde(np.vstack([pc1, pc2]), bw_method='scott')
    dens = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
    dens = np.clip(dens, 1e-10, None)
    G = -kT * np.log(dens)
    G -= G.min()
    return X, Y, G

fel_data = {}
for label, res in pca_results.items():
    X, Y, G = compute_FEL(res['pc1'], res['pc2'])
    fel_data[label] = {'X': X, 'Y': Y, 'G': G}
    print(f'{label}: dG range 0 to {G[G<1e8].max():.2f} kJ/mol')
print('\nFEL computed.')

## Step 6 — Identify Top-5 Lowest-Energy Basins

In [ ]:
from scipy.ndimage import minimum_filter

def find_basins(X, Y, G, n=5):
    nbhd = max(3, int(G.shape[0]*0.05))
    local_min = (G == minimum_filter(G, size=nbhd, mode='reflect'))
    local_min &= (G <= np.percentile(G[G < 1e8], 70))
    rows, cols = np.where(local_min)
    if len(rows) == 0:
        return []
    g_vals = G[rows, cols]
    order  = np.argsort(g_vals)
    rows, cols, g_vals = rows[order], cols[order], g_vals[order]
    basins = []
    for i in range(len(rows)):
        if len(basins) >= n:
            break
        r, c = rows[i], cols[i]
        if not any(np.sqrt((r-b['row'])**2+(c-b['col'])**2) < nbhd for b in basins):
            basins.append({'row':r, 'col':c, 'x':X[r,c], 'y':Y[r,c],
                           'G':g_vals[i], 'rank':len(basins)+1})
    return basins

basin_data = {}
for label, fd in fel_data.items():
    bs = find_basins(fd['X'], fd['Y'], fd['G'])
    basin_data[label] = bs
    print(f'{label}:')
    for b in bs:
        print(f'  Basin {b["rank"]}: PC1={b["x"]:+.2f}  PC2={b["y"]:+.2f}  dG={b["G"]:.3f} kJ/mol')
print('\nBasins identified.')

## Step 7 — Individual FEL Plots (300 dpi, publication quality)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.linewidth': 0.8, 'figure.dpi': 120, 'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})

CMAP_FEL = LinearSegmentedColormap.from_list(
    'FEL', ['#1a3a6b','#2166ac','#74add1','#e0f3f8',
            '#fee090','#f46d43','#d73027','#7f0000'], N=256)
BASIN_COLORS = ['#FFD700','#C0C0C0','#CD7F32','#ADFF2F','#00CED1']
G_MAX = 15.0

def plot_single(label, fd, pr, basins, save=True):
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    X, Y, G = fd['X'], fd['Y'], fd['G']
    ev = pr['ev']
    G_disp = np.clip(G, 0, G_MAX)
    cf = ax.contourf(X, Y, G_disp, levels=200, cmap=CMAP_FEL, extend='max', zorder=1)
    cs = ax.contour(X, Y, G_disp, levels=np.arange(0, G_MAX+1, 1),
                    colors='white', linewidths=0.4, alpha=0.5, zorder=2)
    ax.clabel(cs, levels=[l for l in np.arange(0, G_MAX, 2) if l > 0],
              fmt='%g', fontsize=7, inline=True)
    stride = max(1, len(pr['pc1'])//2000)
    ax.scatter(pr['pc1'][::stride], pr['pc2'][::stride],
               s=1, c='white', alpha=0.08, zorder=3, linewidths=0)
    for i, b in enumerate(basins):
        ax.scatter(b['x'], b['y'], s=160, c=BASIN_COLORS[i], marker='*',
                   edgecolors='k', linewidths=0.5, zorder=10,
                   label=f"B{b['rank']} {b['G']:.2f} kJ/mol")
        ax.annotate(str(b['rank']), xy=(b['x'], b['y']),
                    xytext=(b['x']+0.3, b['y']+0.3),
                    fontsize=8, fontweight='bold', color='white', zorder=11,
                    path_effects=[pe.withStroke(linewidth=1.5, foreground='black')])
    cbar = plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('dG (kJ/mol)', fontsize=10)
    cbar.set_ticks(np.arange(0, G_MAX+1, 3))
    ax.set_xlabel(f'PC1 ({ev[0]:.1f}% var.)')
    ax.set_ylabel(f'PC2 ({ev[1]:.1f}% var.)')
    ax.set_title(label, fontweight='bold')
    if basins:
        ax.legend(title='Energy basins', fontsize=7, title_fontsize=8,
                  loc='upper right', framealpha=0.6)
    plt.tight_layout()
    if save:
        fname = os.path.join(OUTPUT_DIR, f'FEL_{label.replace(" ","_")}.png')
        plt.savefig(fname, dpi=300)
        print(f'  Saved: {fname}')
    plt.show()
    plt.close()

for label in COMPLEXES:
    plot_single(label, fel_data[label], pca_results[label], basin_data[label])
print('Individual plots done.')

## Step 8 — 4-Panel Comparison Figure (Manuscript, shared colorbar)

In [ ]:
def plot_4panel(G_MAX=15.0, save=True):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    cf_ref = None
    levels = np.linspace(0, G_MAX, 200)
    for i, (label, fd) in enumerate(fel_data.items()):
        ax  = axes.flat[i]
        X, Y, G = fd['X'], fd['Y'], fd['G']
        pr  = pca_results[label]
        ev  = pr['ev']
        G_disp = np.clip(G, 0, G_MAX)
        cf = ax.contourf(X, Y, G_disp, levels=levels, cmap=CMAP_FEL, extend='max', zorder=1)
        if i == 0:
            cf_ref = cf
        cs = ax.contour(X, Y, G_disp, levels=np.arange(0, G_MAX+1, 2),
                        colors='white', linewidths=0.5, alpha=0.5, zorder=2)
        ax.clabel(cs, fmt='%g kJ', fontsize=7, inline=True)
        stride = max(1, len(pr['pc1'])//1500)
        ax.scatter(pr['pc1'][::stride], pr['pc2'][::stride],
                   s=0.8, c='white', alpha=0.1, zorder=3, linewidths=0)
        for j, b in enumerate(basin_data[label]):
            ax.scatter(b['x'], b['y'], s=140, c=BASIN_COLORS[j], marker='*',
                       edgecolors='k', linewidths=0.5, zorder=10)
            ax.annotate(str(b['rank']), xy=(b['x'], b['y']),
                        xytext=(b['x']+0.25, b['y']+0.25), fontsize=8,
                        fontweight='bold', color='white', zorder=11,
                        path_effects=[pe.withStroke(linewidth=1.5, foreground='black')])
        ax.set_xlabel(f'PC1 ({ev[0]:.1f}%)', fontsize=10)
        ax.set_ylabel(f'PC2 ({ev[1]:.1f}%)', fontsize=10)
        ax.set_title(label, fontweight='bold', fontsize=12)
        ax.text(-0.10, 1.05, chr(ord('a')+i)+')',
                transform=ax.transAxes, fontsize=13, fontweight='bold', va='top')
    fig.subplots_adjust(right=0.88, hspace=0.35, wspace=0.32)
    cax  = fig.add_axes([0.91, 0.15, 0.022, 0.70])
    cbar = fig.colorbar(cf_ref, cax=cax)
    cbar.set_label('dG (kJ/mol)', fontsize=11)
    cbar.set_ticks(np.arange(0, G_MAX+1, 3))
    fig.suptitle('Free Energy Landscapes of Receptor-De Novo Binder Complexes',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    if save:
        for ext in ('png', 'pdf'):
            p = os.path.join(OUTPUT_DIR, f'FEL_4panel_comparison.{ext}')
            plt.savefig(p, dpi=300, bbox_inches='tight')
            print(f'Saved: {p}')
    plt.show()
    plt.close()

plot_4panel()

## Step 9 — 3D Surface FEL (Supplementary / Presentations)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def plot_3D(G_MAX=12.0, save=True):
    fig = plt.figure(figsize=(14, 11))
    for i, (label, fd) in enumerate(fel_data.items()):
        ax = fig.add_subplot(2, 2, i+1, projection='3d')
        X, Y, G = fd['X'], fd['Y'], fd['G']
        G_disp = np.clip(G, 0, G_MAX)
        s = 3
        ax.plot_surface(X[::s,::s], Y[::s,::s], G_disp[::s,::s],
                        cmap=CMAP_FEL, linewidth=0, antialiased=True,
                        vmin=0, vmax=G_MAX, alpha=0.92)
        for b in basin_data[label]:
            ax.scatter(b['x'], b['y'], b['G']+0.1,
                       c=BASIN_COLORS[b['rank']-1], s=120, marker='*',
                       edgecolors='k', linewidths=0.5)
        ev = pca_results[label]['ev']
        ax.set_xlabel(f'PC1 ({ev[0]:.1f}%)', fontsize=8)
        ax.set_ylabel(f'PC2 ({ev[1]:.1f}%)', fontsize=8)
        ax.set_zlabel('dG (kJ/mol)', fontsize=8)
        ax.set_title(label, fontweight='bold', fontsize=11)
        ax.view_init(elev=30, azim=225)
        ax.set_zlim(0, G_MAX)
    fig.suptitle('3D Free Energy Landscapes', fontsize=12, y=1.01)
    plt.tight_layout()
    if save:
        p = os.path.join(OUTPUT_DIR, 'FEL_3D_surfaces.png')
        plt.savefig(p, dpi=200, bbox_inches='tight')
        print(f'Saved: {p}')
    plt.show()
    plt.close()

plot_3D()

## Step 10 — Stability Metrics Table

In [ ]:
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

def stability_metrics(label, fd, pr, basins, G_thresh=2.0):
    pc1, pc2 = pr['pc1'], pr['pc2']
    X, Y, G  = fd['X'], fd['Y'], fd['G']
    x_vals, y_vals = X[0,:], Y[:,0]
    interp = RegularGridInterpolator((y_vals, x_vals), G,
                                      method='linear', bounds_error=False,
                                      fill_value=np.nan)
    frame_G = interp(np.c_[pc2, pc1])
    frac    = float(np.nanmean(frame_G < G_thresh))
    gap12   = (basins[1]['G'] - basins[0]['G']) if len(basins) >= 2 else float('nan')
    rough   = float(np.std(G[G < np.percentile(G, 80)]))
    return {
        'label': label,
        'Basin 1 DG (kJ/mol)':       round(basins[0]['G'], 3) if basins else float('nan'),
        'Basin 2 DG (kJ/mol)':       round(basins[1]['G'], 3) if len(basins)>=2 else float('nan'),
        'Energy Gap B1-B2 (kJ/mol)': round(gap12, 3) if gap12 == gap12 else float('nan'),
        'Fraction Stable (<2kJ)':    round(frac, 4),
        'Roughness sigma DG':        round(rough, 4),
        'N Interface Residues':      pr['n_iface'],
    }

rows = [stability_metrics(l, fel_data[l], pca_results[l], basin_data[l]) for l in COMPLEXES]
df = pd.DataFrame(rows).set_index('label')
print(df.to_string())
df.to_csv(os.path.join(OUTPUT_DIR, 'stability_metrics.csv'))
print('\nMetrics saved.')

## Step 11 — Stability Bar Chart

In [ ]:
def plot_bars(df, save=True):
    labels  = df.index.tolist()
    n       = len(labels)
    palette = ['#2166ac','#4dac26','#d01c8b','#f1a340']
    metrics = [
        ('Energy Gap B1-B2 (kJ/mol)',  'Energy Gap B2-B1 (kJ/mol)', True),
        ('Fraction Stable (<2kJ)',      'Fraction in Stable Basin',   True),
        ('Roughness sigma DG',          'Landscape Roughness',        False),
        ('Basin 2 DG (kJ/mol)',         'Second Basin dG (kJ/mol)',   True),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
    for ax, (col, ylabel, higher_better) in zip(axes, metrics):
        vals = df[col].values.astype(float)
        bars = ax.bar(range(n), vals, color=palette,
                      edgecolor='white', linewidth=0.8, width=0.6, zorder=3)
        best = int(np.nanargmax(vals) if higher_better else np.nanargmin(vals))
        bars[best].set_edgecolor('black')
        bars[best].set_linewidth(2)
        ax.annotate('BEST', xy=(best, vals[best]),
                    xytext=(best, vals[best]*1.06),
                    ha='center', fontsize=9, fontweight='bold')
        for bar, v in zip(bars, vals):
            if v == v:
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+np.nanmax(vals)*0.01,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(range(n))
        ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
        ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    fig.suptitle('Stability Metrics: 4 Complexes Compared',
                 fontsize=11, y=1.02)
    plt.tight_layout()
    if save:
        for ext in ('png','pdf'):
            p = os.path.join(OUTPUT_DIR, f'stability_bars.{ext}')
            plt.savefig(p, dpi=300, bbox_inches='tight')
            print(f'Saved: {p}')
    plt.show()
    plt.close()

plot_bars(df)

## Step 12 — Extract Representative Basin Structures as PDB

In [ ]:
def save_basin_pdbs(label, traj, pr, basins):
    pc1, pc2 = pr['pc1'], pr['pc2']
    tag = label.replace(' ', '_')
    saved = []
    for b in basins:
        dists = np.sqrt((pc1-b['x'])**2 + (pc2-b['y'])**2)
        frame_idx = int(np.argmin(dists))
        fname = os.path.join(OUTPUT_DIR,
                             f'{tag}_basin{b["rank"]}_frame{frame_idx}.pdb')
        traj[frame_idx].save_pdb(fname)
        saved.append(fname)
        print(f'  {label} | Basin {b["rank"]} | frame {frame_idx} '
              f'PC1={b["x"]:+.2f} PC2={b["y"]:+.2f} '
              f'dG={b["G"]:.3f} -> {os.path.basename(fname)}')
    return saved

basin_pdbs = {}
for label, traj in trajs.items():
    print(f'\n{label}')
    basin_pdbs[label] = save_basin_pdbs(label, traj, pca_results[label], basin_data[label])
print('\nBasin PDBs saved.')

## Step 13 — Interactive 3D Visualization (Basin 1 per complex)

In [ ]:
import nglview as nv
from IPython.display import display

for label, pdbs in basin_pdbs.items():
    if pdbs:
        t = md.load(pdbs[0])
        view = nv.show_mdtraj(t)
        view.add_cartoon(selection='protein', color='chainid')
        view.layout.width  = '700px'
        view.layout.height = '420px'
        print(f'\n{label} - Basin 1 (lowest energy)')
        display(view)

## Step 14 — Composite Stability Ranking

In [ ]:
import textwrap

df_rank = df[['Fraction Stable (<2kJ)',
              'Energy Gap B1-B2 (kJ/mol)',
              'Roughness sigma DG']].copy()
df_rank['score'] = (
    df_rank['Fraction Stable (<2kJ)'].rank(ascending=True) +
    df_rank['Energy Gap B1-B2 (kJ/mol)'].rank(ascending=True) +
    df_rank['Roughness sigma DG'].rank(ascending=False)
)
df_rank = df_rank.sort_values('score', ascending=False)

print('=' * 60)
print('  COMPOSITE STABILITY RANKING')
print('=' * 60)
medals = {1: '[GOLD]', 2: '[SILVER]', 3: '[BRONZE]'}
for rank, (label, row) in enumerate(df_rank.iterrows(), 1):
    medal = medals.get(rank, f'  {rank}.')
    print(f'\n{medal}  {label}')
    print(f'     Fraction stable : {row["Fraction Stable (<2kJ)"]*100:.1f}%')
    print(f'     Energy gap      : {row["Energy Gap B1-B2 (kJ/mol)"]:.3f} kJ/mol')
    print(f'     Roughness       : {row["Roughness sigma DG"]:.4f}')

winner = df_rank.index[0]
print('\n' + '-'*60)
print(textwrap.fill(
    f'RECOMMENDATION: {winner} achieves the deepest, most populated '
    'energy basin with the smoothest landscape, indicating the best '
    'thermodynamic stability among the 4 complexes tested.',
    width=60))
print('-'*60)

## Step 15 — Download All Results

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/FEL_Analysis_Results'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'Archive: {zip_path}.zip')
files.download(f'{zip_path}.zip')

---
## Methods Paragraph (copy for manuscript)

Free energy landscapes (FEL) were constructed from MD trajectories generated with OpenMM. Trajectories were processed using MDTraj. For each complex, receptor backbone Cα atoms and per-complex interface residues (receptor residues within 5 Å of any binder atom, determined from minimum contact distance over the full trajectory) were used as the PCA basis, enabling cross-complex comparison despite differing binder lengths. Coordinates were superposed on the initial frame and standardized prior to PCA (scikit-learn). Free energies were estimated as ΔG = −kᴪT ln P(PC1, PC2), where P was computed by Gaussian kernel density estimation (Scott bandwidth rule; scipy) on the 2D PC space at T = 300 K. The global minimum of each surface was set to zero. Energy basins were identified as local minima using a minimum-filter (5% grid-width neighbourhood), retaining the five deepest after suppressing nearby duplicates.